► Путь фотон испущен из источника с координатами (x,y,z) в направлении (nx,ny,nz). Рзаработать алгоритм, чтобы определить, попадет ли фотон в какой-либо фотоумножитель. Рассмотреть упрощенный случай:
- пренебречь рассеянием и поглощением фотонов в сцинтилляторе
- считать, что фотоны полностью поглощаются, достигая границы детектора

► Пусть количество фотоэлектронов, производимое фотокатодом, подчиняется распределению Пуассона со средним значением a. Реализовать метод, возвращающий случайное число фотоэлектронов. Какому классу должен принадлежать этот метод? Обоснуйте ваш выбор.

► Запустить моделирование для одного события с энергией 1 МэВ в центре детектора. Рассчитать полный собранный заряд (сумму фотоэлектронов по всем фотоумножителям). Насколько это число согласуется с ожидаемым 
`Q = a*Y*(N*r**2/R**2)`

In [1]:
import numpy as np
import matplotlib.pyplot as plt

In [2]:
%run task_1_Detector.ipynb
%run task_2_Source.ipynb

Точка в центре ФЭУ №0 принадлежит ФЭУ? True
Энергия события: 5.0 МэВ
Ожидаемое число фотонов: 50000
Фактически сгенерировано фотонов: 49942


In [3]:
def track_photon_to_boundary(start_pos, direction, detector):

    x0, y0, z0 = start_pos
    nx, ny, nz = direction
    
    a_coef = 1.0
    b_coef = 2.0 * (x0 * nx + y0 * ny + z0 * nz)
    c_coef = (x0 ** 2 + y0 ** 2 + z0 ** 2) - detector.R ** 2
    
    discriminant = b_coef ** 2 - 4.0 * a_coef * c_coef
    
    sqrt_disc = np.sqrt(discriminant)
    t1 = (-b_coef + sqrt_disc) / (2.0 * a_coef)
    t2 = (-b_coef - sqrt_disc) / (2.0 * a_coef)
    
    valid_t = [t for t in (t1, t2) if t > 0]
    t_hit = min(valid_t)
    hit_point = (x0 + t_hit * nx, y0 + t_hit * ny, z0 + t_hit * nz)

    result_is_point_on = detector.is_point_on(hit_point)
    
    return result_is_point_on

In [4]:
def generate_photoelectrons(mean_a):
    photoelectrons = np.random.poisson(mean_a)
    return photoelectrons

In [5]:
E_MeV = 1.0
R = 2.0
N_PMT = 40
r_cm = 10.0
a = 0.5
Y = 10000

In [6]:
r_m = r_cm / 100.0
Q_formula = a * Y * (N_PMT * r_m**2 / R**2)

In [7]:
detector = Detector(R=R, N=N_PMT, r_cm=r_cm, a=a)
source = Source(Y=Y)

In [8]:
directions = source.generate_photons(E_MeV)
start_position = np.array([0.0, 0.0, 0.0])
total_collected_charge = 0

In [9]:
for direction in directions:
    hits_pmt = track_photon_to_boundary(start_position, direction, detector)
    
    if hits_pmt:
        pe_count = generate_photoelectrons(detector.a)
        total_collected_charge += pe_count

In [10]:
print(f"Всего сгенерировано фотонов (среднее {Y*E_MeV}): {len(directions)}")
print(f"Полный собранный заряд (фотоэлектронов): {total_collected_charge}")
print(f"Ожидаемый Q (по формуле из условия): {Q_formula:.2f}")

Всего сгенерировано фотонов (среднее 10000.0): 9816
Полный собранный заряд (фотоэлектронов): 151
Ожидаемый Q (по формуле из условия): 500.00


Что-то вообще не согласуется 😭😭😭